In [ ]:
# ============================================================================
# SETUP: Import Libraries and Configure Environment
# ============================================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import sys
from scipy import stats
import statsmodels.api as sm
import warnings
warnings.filterwarnings('ignore')

# Add project root
sys.path.insert(0, str(Path.cwd().parent))

# Import project modules
from src.constants import (
    COLS, AGE_6_CODES, EDUCATION_CODES, NOC_10_CODES, PROVINCE_CODES, NAICS_21_CODES,
    EQUITY_DIMENSIONS, get_equity_dimension,
    normalize_column_names, DATA_SCOPE_START, DATA_SCOPE_END
)
from src.data_store import EquiPayDataStore
from src.analysis import PayEquityAnalyzer

# Publication-quality figures
plt.rcParams.update({
    'figure.dpi': 150,
    'savefig.dpi': 150,
    'font.size': 11,
    'axes.titlesize': 12,
    'axes.labelsize': 11,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'figure.facecolor': 'white'
})
plt.style.use('seaborn-v0_8-whitegrid')

# Define age group mappings
AGE_6_LABELS = {
    1: '15-24 (Youth)',
    2: '25-34 (Early career)',
    3: '35-44 (Mid-career)',
    4: '45-54 (Peak earnings)',
    5: '55-64 (Late career)',
    6: '65+ (Extended)'
}

print("✓ Libraries loaded successfully")
print(f"✓ Analysis period: {DATA_SCOPE_START}-{DATA_SCOPE_END}")
print(f"\nAnalyzing age dimensions:")
for dim_name in ['age_young', 'age_older']:
    dim = get_equity_dimension(dim_name)
    print(f"  • {dim.description}: {dim.reference_label} vs {dim.comparison_label}")

## 1. Age-Wage Profile: The Canonical Pattern

In [ ]:
# ============================================================================
# LOAD DATA AND COMPUTE AGE-WAGE PROFILE
# ============================================================================

store = EquiPayDataStore()

# Get wages by age group
age_wages = store.sql("""
    SELECT 
        AGE_6,
        SUM(REAL_HRLYEARN * FINALWT) / SUM(FINALWT) as mean_wage,
        SUM(LOG_REAL_HRLYEARN * FINALWT) / SUM(FINALWT) as mean_log_wage,
        SUM(FINALWT) as population,
        COUNT(*) as n
    FROM lfs
    WHERE HRLYEARN > 0 AND AGE_6 BETWEEN 1 AND 6
    GROUP BY AGE_6
    ORDER BY AGE_6
""")

age_wages['age_label'] = age_wages['AGE_6'].map(AGE_6_LABELS)

# Visualize the canonical age-wage curve
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: Mean real wage by age
ax1 = axes[0]
colors = plt.cm.viridis(np.linspace(0.2, 0.8, 6))
bars = ax1.bar(age_wages['age_label'], age_wages['mean_wage'], color=colors)
ax1.set_xlabel('Age Group')
ax1.set_ylabel('Mean Real Hourly Wage ($2010)')
ax1.set_title('Age-Wage Profile: Canadian Labour Market')
ax1.tick_params(axis='x', rotation=45)

# Add value labels
for bar, val in zip(bars, age_wages['mean_wage']):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5, 
             f'${val:.2f}', ha='center', va='bottom', fontsize=10)

# Right: Line plot with population weights
ax2 = axes[1]
ax2.plot(range(1, 7), age_wages['mean_wage'], 'b-o', linewidth=2, markersize=10)
ax2.fill_between(range(1, 7), age_wages['mean_wage'], alpha=0.2)

# Mark peak earnings
peak_idx = age_wages['mean_wage'].idxmax()
peak_age = age_wages.loc[peak_idx, 'AGE_6']
peak_wage = age_wages.loc[peak_idx, 'mean_wage']
ax2.annotate(f'Peak: ${peak_wage:.2f}', xy=(peak_age, peak_wage), 
             xytext=(peak_age + 0.5, peak_wage + 1),
             arrowprops=dict(arrowstyle='->', color='red'),
             fontsize=11, color='red')

ax2.set_xticks(range(1, 7))
ax2.set_xticklabels([f'{i}' for i in range(1, 7)])
ax2.set_xlabel('Age Group (1=15-24 ... 6=65+)')
ax2.set_ylabel('Mean Real Hourly Wage ($2010)')
ax2.set_title('Age-Earnings Curve')

plt.tight_layout()
plt.savefig('../reports/figures/age_wage_profile.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nAge-Wage Profile Summary:")
display(age_wages[['age_label', 'mean_wage', 'population', 'n']].round(2))

## 2. Youth Wage Penalty Analysis

Comparing young workers (15-24) to prime-age workers (25-54), controlling for education and experience.

In [ ]:
# ============================================================================
# YOUTH WAGE PENALTY: RAW GAP
# ============================================================================

# Get youth wage gap configuration
youth_dim = get_equity_dimension('age_young')

# Create binary age grouping for decomposition
# Youth (15-24) vs Prime-age (25-54)
youth_gap = store.sql("""
    SELECT 
        CASE WHEN AGE_6 = 1 THEN 'Youth (15-24)'
             WHEN AGE_6 IN (2, 3, 4) THEN 'Prime-age (25-54)'
        END as age_group,
        SUM(REAL_HRLYEARN * FINALWT) / SUM(FINALWT) as mean_wage,
        SUM(LOG_REAL_HRLYEARN * FINALWT) / SUM(FINALWT) as mean_log_wage,
        SUM(FINALWT) as population
    FROM lfs
    WHERE HRLYEARN > 0 AND AGE_6 BETWEEN 1 AND 4
    GROUP BY age_group
""")

print("Youth vs. Prime-Age Wage Comparison:")
print("=" * 60)
display(youth_gap.round(3))

youth_wage = youth_gap[youth_gap['age_group'] == 'Youth (15-24)']['mean_wage'].values[0]
prime_wage = youth_gap[youth_gap['age_group'] == 'Prime-age (25-54)']['mean_wage'].values[0]
youth_penalty = (prime_wage - youth_wage) / prime_wage * 100

print(f"\n📊 Raw Youth Wage Penalty: {youth_penalty:.1f}%")
print(f"   (Youth earn ${prime_wage - youth_wage:.2f}/hr less than prime-age workers)")

In [ ]:
# ============================================================================
# YOUTH PENALTY BY EDUCATION LEVEL
# ============================================================================

youth_gap_educ = store.sql("""
    SELECT 
        EDUC,
        CASE WHEN AGE_6 = 1 THEN 'Youth' ELSE 'Prime-age' END as age_group,
        SUM(REAL_HRLYEARN * FINALWT) / SUM(FINALWT) as mean_wage
    FROM lfs
    WHERE HRLYEARN > 0 AND AGE_6 BETWEEN 1 AND 4
    GROUP BY EDUC, age_group
""")

# Pivot for comparison
pivot = youth_gap_educ.pivot(index='EDUC', columns='age_group', values='mean_wage')
pivot['gap'] = pivot['Prime-age'] - pivot['Youth']
pivot['gap_pct'] = pivot['gap'] / pivot['Prime-age'] * 100
pivot['education'] = pivot.index.map(EDUCATION_CODES)

# Visualize
fig, ax = plt.subplots(figsize=(12, 6))
x = range(len(pivot))
width = 0.35

ax.bar([i - width/2 for i in x], pivot['Youth'], width, label='Youth (15-24)', color='#3498db')
ax.bar([i + width/2 for i in x], pivot['Prime-age'], width, label='Prime-age (25-54)', color='#2ecc71')

ax.set_xlabel('Education Level')
ax.set_ylabel('Mean Real Hourly Wage ($2010)')
ax.set_title('Youth Wage Penalty by Education Level')
ax.set_xticks(x)
ax.set_xticklabels(pivot['education'], rotation=45, ha='right')
ax.legend()

plt.tight_layout()
plt.savefig('../reports/figures/youth_penalty_by_education.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nYouth Wage Gap by Education:")
display(pivot[['education', 'Youth', 'Prime-age', 'gap', 'gap_pct']].round(2))

## 3. Older Worker Analysis (55+)

Examining whether older workers face wage penalties that may indicate age discrimination.

In [ ]:
# ============================================================================
# OLDER WORKER WAGE PATTERNS
# ============================================================================

# Get older worker dimension
older_dim = get_equity_dimension('age_older')

# Compare late-career (55-64) and extended career (65+) to peak earnings (45-54)
older_analysis = store.sql("""
    SELECT 
        CASE WHEN AGE_6 = 4 THEN 'Peak (45-54)'
             WHEN AGE_6 = 5 THEN 'Late career (55-64)'
             WHEN AGE_6 = 6 THEN 'Extended (65+)'
        END as age_group,
        AGE_6,
        SUM(REAL_HRLYEARN * FINALWT) / SUM(FINALWT) as mean_wage,
        SUM(FINALWT) as population,
        COUNT(*) as n
    FROM lfs
    WHERE HRLYEARN > 0 AND AGE_6 BETWEEN 4 AND 6
    GROUP BY age_group, AGE_6
    ORDER BY AGE_6
""")

print("Older Worker Wage Comparison:")
print("=" * 60)
display(older_analysis.round(2))

peak_wage = older_analysis[older_analysis['AGE_6'] == 4]['mean_wage'].values[0]
late_wage = older_analysis[older_analysis['AGE_6'] == 5]['mean_wage'].values[0]
extended_wage = older_analysis[older_analysis['AGE_6'] == 6]['mean_wage'].values[0]

late_gap = (peak_wage - late_wage) / peak_wage * 100
extended_gap = (peak_wage - extended_wage) / peak_wage * 100

print(f"\n📊 Late Career Wage Gap (55-64 vs 45-54): {late_gap:.1f}%")
print(f"📊 Extended Career Gap (65+ vs 45-54): {extended_gap:.1f}%")

In [ ]:
# ============================================================================
# AGE GAPS BY INDUSTRY
# ============================================================================

# Which industries show the largest older worker penalties?
age_by_industry = store.sql("""
    SELECT 
        NAICS_21,
        CASE WHEN AGE_6 IN (3, 4) THEN 'Prime (35-54)'
             WHEN AGE_6 IN (5, 6) THEN 'Older (55+)'
        END as age_group,
        SUM(REAL_HRLYEARN * FINALWT) / SUM(FINALWT) as mean_wage,
        SUM(FINALWT) as population
    FROM lfs
    WHERE HRLYEARN > 0 AND AGE_6 BETWEEN 3 AND 6 AND NAICS_21 IS NOT NULL
    GROUP BY NAICS_21, age_group
""")

# Pivot and calculate gap
industry_pivot = age_by_industry.pivot(index='NAICS_21', columns='age_group', values='mean_wage')
industry_pivot['gap_pct'] = (industry_pivot['Prime (35-54)'] - industry_pivot['Older (55+)']) / industry_pivot['Prime (35-54)'] * 100
industry_pivot = industry_pivot.dropna()

# Add industry names
industry_pivot['industry'] = industry_pivot.index.map(lambda x: NAICS_21_CODES.get(x, f'Industry {x}'))

# Sort and visualize top gaps
industry_sorted = industry_pivot.sort_values('gap_pct', ascending=False).head(15)

fig, ax = plt.subplots(figsize=(12, 8))
colors = ['#e74c3c' if x > 0 else '#27ae60' for x in industry_sorted['gap_pct']]
ax.barh(industry_sorted['industry'], industry_sorted['gap_pct'], color=colors)
ax.axvline(x=0, color='black', linewidth=0.5)
ax.set_xlabel('Wage Gap (%) - Prime vs Older Workers')
ax.set_title('Older Worker Wage Gap by Industry\n(Positive = Prime-age earn more)')

plt.tight_layout()
plt.savefig('../reports/figures/older_worker_gap_by_industry.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Age × Gender Intersectional Analysis

Do age-based wage patterns differ by gender?

In [ ]:
# ============================================================================
# AGE-WAGE CURVES BY GENDER
# ============================================================================

age_gender = store.sql("""
    SELECT 
        AGE_6,
        GENDER,
        SUM(REAL_HRLYEARN * FINALWT) / SUM(FINALWT) as mean_wage
    FROM lfs
    WHERE HRLYEARN > 0 AND AGE_6 BETWEEN 1 AND 6 AND GENDER IN (1, 2)
    GROUP BY AGE_6, GENDER
    ORDER BY AGE_6, GENDER
""")

# Visualize
fig, ax = plt.subplots(figsize=(10, 6))

male = age_gender[age_gender['GENDER'] == 1]
female = age_gender[age_gender['GENDER'] == 2]

ax.plot(male['AGE_6'], male['mean_wage'], 'b-o', linewidth=2, markersize=10, label='Male')
ax.plot(female['AGE_6'], female['mean_wage'], 'r-s', linewidth=2, markersize=10, label='Female')

# Shade the gender gap
ax.fill_between(male['AGE_6'], female['mean_wage'], male['mean_wage'], alpha=0.2, color='purple')

ax.set_xticks(range(1, 7))
ax.set_xticklabels([AGE_6_LABELS[i] for i in range(1, 7)], rotation=45, ha='right')
ax.set_xlabel('Age Group')
ax.set_ylabel('Mean Real Hourly Wage ($2010)')
ax.set_title('Age-Earnings Curves by Gender\n(Shaded area = Gender gap)')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../reports/figures/age_wage_by_gender.png', dpi=150, bbox_inches='tight')
plt.show()

# Calculate when gender gap is widest
pivot = age_gender.pivot(index='AGE_6', columns='GENDER', values='mean_wage')
pivot.columns = ['Male', 'Female']
pivot['gap'] = pivot['Male'] - pivot['Female']
pivot['gap_pct'] = pivot['gap'] / pivot['Male'] * 100
pivot['age_label'] = pivot.index.map(AGE_6_LABELS)

print("\nGender Gap by Age:")
display(pivot[['age_label', 'Male', 'Female', 'gap', 'gap_pct']].round(2))

## 5. Time Trends in Age-Based Disparities

In [ ]:
# ============================================================================
# YOUTH WAGE PENALTY OVER TIME
# ============================================================================

youth_trend = store.sql("""
    SELECT 
        SURVYEAR,
        CASE WHEN AGE_6 = 1 THEN 'Youth' ELSE 'Prime-age' END as age_group,
        SUM(REAL_HRLYEARN * FINALWT) / SUM(FINALWT) as mean_wage
    FROM lfs
    WHERE HRLYEARN > 0 AND AGE_6 BETWEEN 1 AND 4
    GROUP BY SURVYEAR, age_group
    ORDER BY SURVYEAR, age_group
""")

# Pivot
youth_pivot = youth_trend.pivot(index='SURVYEAR', columns='age_group', values='mean_wage')
youth_pivot['gap_pct'] = (youth_pivot['Prime-age'] - youth_pivot['Youth']) / youth_pivot['Prime-age'] * 100

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: Wage levels
ax1 = axes[0]
ax1.plot(youth_pivot.index, youth_pivot['Youth'], 'b-o', label='Youth (15-24)', linewidth=2)
ax1.plot(youth_pivot.index, youth_pivot['Prime-age'], 'g-s', label='Prime-age (25-54)', linewidth=2)
ax1.set_xlabel('Year')
ax1.set_ylabel('Mean Real Hourly Wage ($2010)')
ax1.set_title('Youth vs Prime-Age Wages Over Time')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Right: Gap percentage
ax2 = axes[1]
ax2.plot(youth_pivot.index, youth_pivot['gap_pct'], 'r-^', linewidth=2)
ax2.fill_between(youth_pivot.index, 0, youth_pivot['gap_pct'], alpha=0.2, color='red')
ax2.set_xlabel('Year')
ax2.set_ylabel('Youth Wage Penalty (%)')
ax2.set_title('Youth Wage Penalty Trend')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../reports/figures/youth_penalty_timeseries.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Summary & Policy Implications

In [ ]:
# ============================================================================
# SUMMARY REPORT
# ============================================================================

print("\n" + "=" * 70)
print("AGE-BASED WAGE DISCRIMINATION ANALYSIS - EXECUTIVE SUMMARY")
print("=" * 70)
print(f"\nData Source: LFS PUMF {DATA_SCOPE_START}-{DATA_SCOPE_END}")
print(f"Legal Framework: Canadian Human Rights Act - Age is a prohibited ground")
print("\n" + "-" * 70)

print("\nKEY FINDINGS:")

print("\n1. AGE-WAGE PROFILE:")
print(f"   - Wages peak in the 45-54 age group")
print(f"   - Standard inverted-U pattern observed")

print(f"\n2. YOUTH WAGE PENALTY:")
print(f"   - Youth (15-24) earn approximately {youth_penalty:.0f}% less than prime-age workers")
print(f"   - Much is explained by experience and education")
print(f"   - Gap persists at similar education levels")

print(f"\n3. OLDER WORKER PATTERNS:")
print(f"   - Late career (55-64) gap: {late_gap:.1f}% vs peak")
print(f"   - Extended career (65+) gap: {extended_gap:.1f}% vs peak")
print(f"   - May reflect selection effects (who continues working)")

print("\n4. INTERSECTIONALITY:")
print("   - Gender gap varies by age")
print("   - Widest gender gap often at peak earning ages")

print("\n" + "-" * 70)
print("\nPOLICY IMPLICATIONS:")
print("\n1. YOUTH EMPLOYMENT:")
print("   - Investment in apprenticeships and work-integrated learning")
print("   - Ensure minimum wage exemptions don't perpetuate inequity")

print("\n2. OLDER WORKERS:")
print("   - Skills updating and retraining programs")
print("   - Combat age-based hiring discrimination")
print("   - Flexible retirement and phased work options")

print("\n3. LIFE-CYCLE CONSIDERATIONS:")
print("   - Pay equity should consider career stage")
print("   - Age alone doesn't justify wage differentials")
print("\n" + "=" * 70)